In [1]:
!pip install -q "boxmot==17.0.0" ultralytics supervision

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 4.1 MB/s eta 0:00:0000:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 1.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.2/88.2 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 26.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 37.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 397.5/397.5 kB 9.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 49.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 70.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.3/144.3 kB 

In [2]:
#!/usr/bin/env python3
"""
CODE2 SC_mcmt_motion_kaggle.py — MCMT via MOTION-PREDICTION handoff + overlapping
camera fields of view (supersedes SC_mcmt_kaggle.py's geometric-threshold
approach).

WHY THIS VERSION
  The pure appearance-gallery + geometric-threshold-bonus approach struggles
  when cameras DON'T overlap: a person gets clipped right at the boundary in
  both quadrants at once, giving a bad embedding exactly when you need a good
  one. This version fixes that two ways:

    1) CAMERAS NOW OVERLAP. Each quadrant's detection/tracking runs on a
       slightly LARGER crop (its "full" region: core quadrant + OVERLAP_PX
       into each neighbor), so a person is seen whole by whichever camera's
       core they're actually standing in, checked AGAINST the full region
       each neighbor is detecting with — before they're clipped by that
       neighbor's core boundary. Drawing, zone logic (entry/seller/PEC/
       ignore) and counting stay scoped to each camera's CORE region only
       (so nobody is drawn or counted twice in the shared strip) — but
       tracking/motion/identity-matching runs on the full region.

    2) MOTION PREDICTION IS THE PRIMARY cross-camera matcher, not appearance.
       Each local track's velocity (in shared canvas coordinates) is kept
       live. When a brand-new local track appears anywhere, it's first
       checked against every ADJACENT camera's currently-live tracks: if
       that track's last position, extrapolated forward by elapsed time
       using its velocity, lands close to where the new track started, it's
       the same physical person — no appearance comparison needed. OSNet
       embeddings only step in: (a) as a TIE-BREAK when more than one motion
       candidate matches, and (b) as a FALLBACK (stricter threshold) when no
       motion candidate is found at all (e.g. re-entry after a longer gap).

BEFORE YOU RUN
  - UNIFORM_BAND_LO/HI are baked in from the ORIGINAL CRK seller's uniform
    colour and will likely need re-picking for a different test video.
  - CAMERA_*_ZONES are all empty by default — call
    export_quadrant_reference_frames() once first, read off pixel
    coordinates from the saved PNGs (these are CORE-region crops, no
    overlap), and fill in the CAMERA_* dicts below in those LOCAL
    (core-quadrant) coordinates.
  - boxmot's internal API has moved around across releases; this is written
    against boxmot==17.0.0 — install with:
        !pip install -q "boxmot==17.0.0" ultralytics supervision
    and RESTART THE KERNEL if boxmot was already imported this session,
    before running this file.
"""

import subprocess
import sys


def _pip_install(*packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])


try:
    import boxmot
    if getattr(boxmot, "__version__", "") != "17.0.0":
        _pip_install("boxmot==17.0.0")
except ImportError:
    _pip_install("boxmot==17.0.0")

try:
    import supervision  # noqa: F401
except ImportError:
    _pip_install("supervision")

try:
    import ultralytics  # noqa: F401
except ImportError:
    _pip_install("ultralytics")

import os
import inspect
from collections import namedtuple
from pathlib import Path
from typing import List, Optional, Dict, Tuple

import cv2
import numpy as np
np.float = float  # compat shim for older boxmot/numpy

import torch
from ultralytics import YOLO
import supervision as sv
from boxmot import BotSort

# =====================================================================
# CONFIG
# =====================================================================
SOURCE_VIDEO_PATH = "/kaggle/input/datasets/maramkouki/videotest/part2_40-00_end.mp4"
OUTPUT_DIR = "/kaggle/working"
os.makedirs(OUTPUT_DIR, exist_ok=True)
TARGET_VIDEO_PATH = os.path.join(OUTPUT_DIR, "SC_mcmt_motion_output.mp4")

GRID_ROWS, GRID_COLS = 2, 2    # the "+" cut: TL=0 TR=1 / BL=2 BR=3
NUM_CAMERAS = GRID_ROWS * GRID_COLS
OVERLAP_PX = 100                # how far each camera's DETECTION crop extends into its neighbors

MODEL_NAME = "yolo11m-pose.pt"
REID_MODEL = "osnet_ain_x1_0_msmt17.pt"
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
HALF_PRECISION = torch.cuda.is_available()

DETECT_CONF = 0.07
PERSON_CLASS_ID = 0
SKIP = 2
MAX_FRAMES = 0    # >0: stop after this many PROCESSED frames (quick test run)

# --- BoT-SORT / ReID (per-camera local tracking) ---
TRACK_BUFFER = 180
APPEARANCE_THRESH = 0.3
PROXIMITY_THRESH = 0.8
MATCH_THRESH = 0.8
CMC_METHOD = "sof"
TRACK_HIGH_THRESH = 0.4
TRACK_LOW_THRESH = 0.1
NEW_TRACK_THRESH = 0.2

# --- SELLER cue: baked-in uniform HSV band — re-pick for your test video ---
UNIFORM_BAND_LO = np.array([18, 0, 51])
UNIFORM_BAND_HI = np.array([49, 54, 240])
LOWER_KP_IDX = [11, 12, 13, 14]
UPPER_KP_IDX = [5, 6]
KP_CONF = 0.07
KP_PATCH_R = 3
KP_PATCH_FRAC = 0.5
BEIGE_KP_MIN = 1
WHITE_KP_MIN = 1
WHITE_PATCH_FRAC = 0.5
WHITE_S_MAX = 80
WHITE_V_MIN = 110
SELLER_ZONE_FRAC = 0.50

# --- entry zones (id-free, ORDERED TWO-ZONE pass) ---
ENTER_ZONE_FRAC = 0.10
ZONE_MATCH_DIST = 200
ZONE_TTL = 10
ENTRY_REVERT_SECS = 2.9
REVERT_MIN_SELLER_SECS = 0.5

# --- interaction / PEC ---
INTERACTION_RADIUS = 200
INTERACTION_OVERLAP_MIN = 0.35
INTERACTION_MIN_SECS = 5
INTERACTION_GRACE_SECS = 2

# --- ignore zones ---
IGNORE_ZONE_FRAC = 0.1

# --- MCMT: motion-prediction handoff (primary) ---
POSITION_TOLERANCE_PX = 130      # base match radius; grows with TOL_GROWTH_PX_PER_SEC over time
TOL_GROWTH_PX_PER_SEC = 80       # extra slack added to the match radius per second since last seen
HANDOFF_WINDOW_SEC = 2.5         # max time since last seen, for a motion prediction to still apply
VEL_EMA_ALPHA = 0.6              # velocity smoothing (higher = smoother, slower to react to turns)
BOUNDARY_MARGIN_PX = 110         # "near the shared edge" margin, in CORE-local px, on BOTH sides
MIN_EXIT_SPEED = 8.0             # px/sec of motion THROUGH the shared edge required to be a candidate
                                  # (filters out people merely standing near a boundary)

# --- MCMT: appearance tie-break / fallback (secondary) ---
EMBEDDING_TIEBREAK_MIN = 0.35    # min cosine-sim to trust embeddings over plain nearest-position
FALLBACK_SIM_THRESHOLD = 0.68    # stricter bar for a pure-appearance match (no motion candidate at all)
GLOBAL_GALLERY_TTL_SEC = 25.0
GLOBAL_EMA_ALPHA = 0.85
STATE_CONF_MIN = 0.30            # mean keypoint confidence needed to TRUST this frame's embedding

# --- per-camera zones, in CORE (non-overlap quadrant) LOCAL pixel coords ---
# camera_id layout (row-major): 0=TL  1=TR
#                                2=BL  3=BR
CAMERA_ENTER_ZONE_1: Dict[int, list] = {0: [], 1: [], 2: [], 3: []}
CAMERA_ENTER_ZONE_2: Dict[int, list] = {0: [], 1: [], 2: [], 3: []}
CAMERA_SELLER_ZONES: Dict[int, list] = {0: [], 1: [], 2: [], 3: []}
CAMERA_MIRROR_ZONES: Dict[int, list] = {0: [], 1: [], 2: [], 3: []}
CAMERA_IGNORE_ZONES: Dict[int, list] = {0: [], 1: [], 2: [], 3: []}

print(f"[DEVICE] running on {DEVICE}")


def export_quadrant_reference_frames(frame_number: int = 0, out_dir: str = OUTPUT_DIR):
    """Dump one reference PNG per camera's CORE (non-overlap) region, so you
    can read off pixel coordinates for the CAMERA_*_ZONES dicts above."""
    cap = cv2.VideoCapture(SOURCE_VIDEO_PATH)
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_number)
    ok, frame = cap.read()
    cap.release()
    if not ok:
        print("[export] could not read that frame")
        return
    core_regions = split_regions(frame.shape[1], frame.shape[0], GRID_ROWS, GRID_COLS)
    for idx, ((ox, oy), (w, h)) in enumerate(core_regions):
        crop = frame[oy:oy + h, ox:ox + w]
        path = os.path.join(out_dir, f"cam{idx}_core_reference.png")
        cv2.imwrite(path, crop)
        print(f"[export] camera {idx} CORE: {path}  size={w}x{h}  origin=({ox},{oy})")


# =====================================================================
# Geometry helpers
# =====================================================================
def split_regions(frame_w, frame_h, rows, cols):
    """Non-overlapping CORE regions — one per camera, tiling the whole frame."""
    regions = []
    cw, ch = frame_w // cols, frame_h // rows
    for r in range(rows):
        for c in range(cols):
            ox, oy = c * cw, r * ch
            w = cw if c < cols - 1 else frame_w - ox
            h = ch if r < rows - 1 else frame_h - oy
            regions.append(((ox, oy), (w, h)))
    return regions


def build_adjacency(rows, cols) -> Dict[int, Dict[int, str]]:
    """camera_id -> {neighbor_id: shared_edge_name_on_THIS_camera}."""
    adj = {r * cols + c: {} for r in range(rows) for c in range(cols)}
    for r in range(rows):
        for c in range(cols):
            idx = r * cols + c
            if c + 1 < cols:
                nb = r * cols + (c + 1)
                adj[idx][nb] = 'right'
                adj[nb][idx] = 'left'
            if r + 1 < rows:
                nb = (r + 1) * cols + c
                adj[idx][nb] = 'bottom'
                adj[nb][idx] = 'top'
    return adj


def build_full_regions(core_regions, adjacency, overlap, frame_w, frame_h):
    """Expand each CORE region outward by `overlap` px, but ONLY on edges
    shared with a neighbor (never past the real frame boundary). This is the
    region each camera actually runs detection/tracking on."""
    full = []
    for idx, (origin, size) in enumerate(core_regions):
        ox, oy = origin
        w, h = size
        x1, y1, x2, y2 = ox, oy, ox + w, oy + h
        edges = set(adjacency.get(idx, {}).values())
        if 'left' in edges:
            x1 = max(0, x1 - overlap)
        if 'right' in edges:
            x2 = min(frame_w, x2 + overlap)
        if 'top' in edges:
            y1 = max(0, y1 - overlap)
        if 'bottom' in edges:
            y2 = min(frame_h, y2 + overlap)
        full.append(((x1, y1), (x2 - x1, y2 - y1)))
    return full


def box_overlap_frac(bbox, polys) -> float:
    if not polys:
        return 0.0
    x1, y1, x2, y2 = (float(v) for v in bbox)
    if x2 <= x1 or y2 <= y1:
        return 0.0
    inside = total = 0
    for px in np.linspace(x1, x2, 6):
        for py in np.linspace(y1, y2, 6):
            total += 1
            if any(cv2.pointPolygonTest(poly, (px, py), False) >= 0 for poly in polys):
                inside += 1
    return inside / total if total else 0.0


def circle_overlap_frac(ca, cb, r) -> float:
    d = float(np.hypot(ca[0] - cb[0], ca[1] - cb[1]))
    if d >= 2 * r:
        return 0.0
    if d <= 0:
        return 1.0
    lens = 2 * r * r * np.arccos(d / (2 * r)) - (d / 2) * np.sqrt(max(0.0, 4 * r * r - d * d))
    return float(lens / (np.pi * r * r))


def _feet_in_polys(xyxy: np.ndarray, polys) -> np.ndarray:
    if not polys or len(xyxy) == 0:
        return np.zeros((len(xyxy),), dtype=bool)
    feet = np.c_[(xyxy[:, 0] + xyxy[:, 2]) / 2.0, xyxy[:, 3]]
    mask = np.zeros((len(xyxy),), dtype=bool)
    for i, (fx, fy) in enumerate(feet):
        mask[i] = any(cv2.pointPolygonTest(p, (float(fx), float(fy)), False) >= 0 for p in polys)
    return mask


def seller_zone_frac(bbox, polys) -> float:
    return box_overlap_frac(bbox, polys)


def beige_at_keypoints(frame, kp_xy_p, kp_cf_p) -> bool:
    H, W = frame.shape[:2]
    n_beige = 0
    for idx in LOWER_KP_IDX:
        if kp_cf_p[idx] < KP_CONF:
            continue
        x, y = int(kp_xy_p[idx][0]), int(kp_xy_p[idx][1])
        x1, y1 = max(0, x - KP_PATCH_R), max(0, y - KP_PATCH_R)
        x2, y2 = min(W, x + KP_PATCH_R + 1), min(H, y + KP_PATCH_R + 1)
        patch = frame[y1:y2, x1:x2]
        if patch.size == 0:
            continue
        hsv = cv2.cvtColor(patch, cv2.COLOR_BGR2HSV)
        inband = np.all((hsv >= UNIFORM_BAND_LO) & (hsv <= UNIFORM_BAND_HI), axis=2)
        if inband.mean() >= KP_PATCH_FRAC:
            n_beige += 1
    return n_beige >= BEIGE_KP_MIN


def white_at_keypoints(frame, kp_xy_p, kp_cf_p) -> bool:
    H, W = frame.shape[:2]
    n_white = 0
    for idx in UPPER_KP_IDX:
        if kp_cf_p[idx] < KP_CONF:
            continue
        x, y = int(kp_xy_p[idx][0]), int(kp_xy_p[idx][1])
        x1, y1 = max(0, x - KP_PATCH_R), max(0, y - KP_PATCH_R)
        x2, y2 = min(W, x + KP_PATCH_R + 1), min(H, y + KP_PATCH_R + 1)
        patch = frame[y1:y2, x1:x2]
        if patch.size == 0:
            continue
        hsv = cv2.cvtColor(patch, cv2.COLOR_BGR2HSV)
        white = (hsv[:, :, 1] <= WHITE_S_MAX) & (hsv[:, :, 2] >= WHITE_V_MIN)
        if white.mean() >= WHITE_PATCH_FRAC:
            n_white += 1
    return n_white >= WHITE_KP_MIN


# =====================================================================
# BoT-SORT builder — version-robust (see module docstring for the pin note)
# =====================================================================
_EMB_WARNED = [False]


def build_botsort():
    wanted = dict(
        reid_weights=Path(REID_MODEL), device=torch.device(DEVICE), half=HALF_PRECISION,
        per_class=False, track_high_thresh=TRACK_HIGH_THRESH, track_low_thresh=TRACK_LOW_THRESH,
        new_track_thresh=NEW_TRACK_THRESH, track_buffer=TRACK_BUFFER, match_thresh=MATCH_THRESH,
        proximity_thresh=PROXIMITY_THRESH, appearance_thresh=APPEARANCE_THRESH, cmc_method=CMC_METHOD,
    )
    sig = inspect.signature(BotSort.__init__)
    accepts_var_kwargs = any(p.kind == inspect.Parameter.VAR_KEYWORD for p in sig.parameters.values())
    valid = wanted if accepts_var_kwargs else {k: v for k, v in wanted.items() if k in sig.parameters}
    dropped = sorted(set(wanted) - set(valid))
    if dropped:
        print(f"[botsort] this boxmot version's BotSort doesn't take: {dropped} — using its defaults.")
    return BotSort(**valid)


def extract_embeddings(tracker, boxes: np.ndarray, frame: np.ndarray) -> Optional[np.ndarray]:
    """Best-effort ReID embedding extraction across boxmot API variants."""
    if len(boxes) == 0:
        return np.zeros((0, 1), dtype=float)
    for attr_path, method in (("model", "get_features"), ("reid_model", "get_features"),
                               ("appearance_model", "get_features"), ("model", "__call__")):
        obj = tracker
        ok = True
        for part in attr_path.split("."):
            if not hasattr(obj, part):
                ok = False
                break
            obj = getattr(obj, part)
        if not ok or not hasattr(obj, method):
            continue
        try:
            feats = np.asarray(getattr(obj, method)(boxes, frame)).reshape(len(boxes), -1)
            return feats
        except Exception:
            continue
    if not _EMB_WARNED[0]:
        print("[WARN] no ReID embedding method found on this boxmot tracker — "
              "MCMT will rely on motion prediction only (no appearance tie-break/fallback).")
        _EMB_WARNED[0] = True
    return None


# =====================================================================
# MCMT identity fusion: motion-prediction primary, appearance secondary
# =====================================================================
class MotionIDManager:
    def __init__(self, adjacency, position_tol, handoff_window, vel_ema,
                 tiebreak_min_sim, fallback_sim, gallery_ttl, gallery_ema):
        self.adjacency = adjacency
        self.position_tol = position_tol
        self.handoff_window = handoff_window
        self.vel_ema = vel_ema
        self.tiebreak_min_sim = tiebreak_min_sim
        self.fallback_sim = fallback_sim
        self.gallery_ttl = gallery_ttl
        self.gallery_ema = gallery_ema

        self.local_to_global: Dict[Tuple[int, int], int] = {}
        self.live_states: Dict[Tuple[int, int], dict] = {}
        self.gallery: Dict[int, dict] = {}
        self.next_global_id = 1

    def resolve(self, camera_id: int, tid: Optional[int], global_center: Tuple[float, float],
                embedding: Optional[np.ndarray], timestamp: float) -> int:
        local_key = (camera_id, tid) if tid is not None else None

        if local_key is not None and local_key in self.local_to_global:
            gid = self.local_to_global[local_key]
            self._touch(local_key, global_center, timestamp, gid)
            if embedding is not None:
                self._update_gallery(gid, embedding, timestamp, camera_id)
            return gid

        # --- PRIMARY: motion-predicted hand-off from an adjacent camera ---
        nbrs = self.adjacency.get(camera_id, {})
        candidates = []
        for (cam2, tid2), state in self.live_states.items():
            if cam2 == camera_id or cam2 not in nbrs:
                continue
            dt = timestamp - state['last_ts']
            if dt < 0 or dt > self.handoff_window:
                continue
            pred_x = state['pos'][0] + state['vel'][0] * dt
            pred_y = state['pos'][1] + state['vel'][1] * dt
            err = float(np.hypot(global_center[0] - pred_x, global_center[1] - pred_y))
            if err <= self.position_tol:
                candidates.append((err, state['global_id']))

        gid = None
        if candidates:
            if len(candidates) == 1 or embedding is None:
                candidates.sort(key=lambda c: c[0])
                gid = candidates[0][1]
            else:
                best_sim, best_gid = -1.0, None
                for _err, gid2 in candidates:
                    ent = self.gallery.get(gid2)
                    sim = float(embedding @ ent['emb']) if ent and ent['emb'] is not None else -1.0
                    if sim > best_sim:
                        best_sim, best_gid = sim, gid2
                if best_sim >= self.tiebreak_min_sim:
                    gid = best_gid
                else:
                    candidates.sort(key=lambda c: c[0])
                    gid = candidates[0][1]

        # --- FALLBACK: pure appearance match against the whole gallery ---
        if gid is None and embedding is not None:
            best_gid, best_sim = None, -1.0
            for gid2, entry in self.gallery.items():
                if entry['emb'] is None or timestamp - entry['last_ts'] > self.gallery_ttl:
                    continue
                sim = float(embedding @ entry['emb'])
                if sim >= self.fallback_sim and sim > best_sim:
                    best_sim, best_gid = sim, gid2
            gid = best_gid

        if gid is None:
            gid = self.next_global_id
            self.next_global_id += 1
            self.gallery[gid] = {'emb': None, 'last_ts': timestamp, 'last_camera': camera_id,
                                  'cameras': {camera_id}}

        if local_key is not None:
            self.local_to_global[local_key] = gid
        self._touch(local_key, global_center, timestamp, gid)
        if embedding is not None:
            self._update_gallery(gid, embedding, timestamp, camera_id)
        return gid

    def _touch(self, local_key, global_center, timestamp, gid):
        if local_key is None:
            return
        prev = self.live_states.get(local_key)
        if prev is not None and timestamp > prev['last_ts']:
            dtp = timestamp - prev['last_ts']
            raw_vx = (global_center[0] - prev['pos'][0]) / dtp
            raw_vy = (global_center[1] - prev['pos'][1]) / dtp
            vx = self.vel_ema * prev['vel'][0] + (1 - self.vel_ema) * raw_vx
            vy = self.vel_ema * prev['vel'][1] + (1 - self.vel_ema) * raw_vy
        else:
            vx, vy = 0.0, 0.0
        self.live_states[local_key] = {'pos': global_center, 'vel': (vx, vy),
                                        'last_ts': timestamp, 'global_id': gid}

    def _update_gallery(self, gid, embedding, timestamp, camera_id):
        entry = self.gallery[gid]
        if entry['emb'] is None:
            entry['emb'] = embedding.copy()
        else:
            entry['emb'] = self.gallery_ema * entry['emb'] + (1 - self.gallery_ema) * embedding
            entry['emb'] /= (np.linalg.norm(entry['emb']) + 1e-9)
        entry['last_ts'] = timestamp
        entry['last_camera'] = camera_id
        entry['cameras'].add(camera_id)

    def cleanup(self, timestamp):
        stale = [k for k, s in self.live_states.items() if timestamp - s['last_ts'] > self.handoff_window * 3]
        for k in stale:
            del self.live_states[k]

    def cross_camera_identities(self) -> List[int]:
        return [gid for gid, e in self.gallery.items() if len(e['cameras']) > 1]


Person = namedtuple("Person", "bbox role tid global_id kp_xy kp_cf")


# =====================================================================
# Drawing
# =====================================================================
def draw_frame(crop, persons: List[Person], polys, served_gids):
    enter1, enter2, seller_p, mirror_p, ignore_p = polys
    SELLER_COLOR, CLIENT_COLOR = (0, 0, 255), (0, 200, 0)
    for p in persons:
        x1, y1, x2, y2 = map(int, p.bbox)
        color = SELLER_COLOR if p.role == "SELLER" else CLIENT_COLOR
        label = f"{p.role} G{p.global_id}"
        cv2.rectangle(crop, (x1, y1), (x2, y2), color, 2)
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 2)
        cv2.rectangle(crop, (x1, y1 - th - 8), (x1 + tw + 4, y1), color, -1)
        cv2.putText(crop, label, (x1 + 2, y1 - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2)
        if p.role == "CLIENT" and p.global_id in served_gids:
            cv2.putText(crop, "+1", (x1 + 2, y2 - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
    for poly in mirror_p:
        cv2.polylines(crop, [poly], True, (255, 0, 255), 2, cv2.LINE_AA)
    for poly in seller_p:
        cv2.polylines(crop, [poly], True, (0, 0, 255), 2, cv2.LINE_AA)
    for poly in ignore_p:
        cv2.polylines(crop, [poly], True, (128, 128, 128), 2, cv2.LINE_AA)
    for poly in enter1:
        cv2.polylines(crop, [poly], True, (0, 255, 255), 2, cv2.LINE_AA)
    for poly in enter2:
        cv2.polylines(crop, [poly], True, (255, 200, 0), 2, cv2.LINE_AA)
    return crop


# =====================================================================
# CameraWorker — one per quadrant; detects on FULL (overlap) region,
# draws/counts/zones only within its CORE (non-overlap) region.
# =====================================================================
class CameraWorker:
    def __init__(self, camera_id: int, core_origin, core_size, full_origin, full_size):
        self.camera_id = camera_id
        self.ox, self.oy = core_origin
        self.w, self.h = core_size
        self.fox, self.foy = full_origin
        self.fw, self.fh = full_size
        self.delta = (self.fox - self.ox, self.foy - self.oy)   # full-local -> core-local offset

        self.botsort = build_botsort()

        def _polys(d):
            return [np.array(z, np.int32) for z in d.get(camera_id, []) if len(z) >= 3]

        self.enter1 = _polys(CAMERA_ENTER_ZONE_1)
        self.enter2 = _polys(CAMERA_ENTER_ZONE_2)
        self.seller_polys = _polys(CAMERA_SELLER_ZONES)
        self.mirror_polys = _polys(CAMERA_MIRROR_ZONES)
        self.ignore_polys = _polys(CAMERA_IGNORE_ZONES)

        self.zone_objs: list = []
        self.entry_by_tid: Dict[int, float] = {}
        self.entry_seller_secs: Dict[int, float] = {}
        self.entry_count_local = 0

        self.contact_secs: Dict[int, float] = {}
        self.contact_last_frame: Dict[int, int] = {}

        self.last_core_persons: List[Person] = []

    def process(self, model, canvas, processed_frame_idx, timestamp, fps,
                motion_mgr: MotionIDManager, global_entered_ids: set,
                global_pec_pairs: set, served_gids: set, is_skip: bool):
        full_view = canvas[self.foy:self.foy + self.fh, self.fox:self.fox + self.fw]
        core_view = canvas[self.oy:self.oy + self.h, self.ox:self.ox + self.w]
        polys = (self.enter1, self.enter2, self.seller_polys, self.mirror_polys, self.ignore_polys)

        if is_skip:
            draw_frame(core_view, self.last_core_persons, polys, served_gids)
            return

        results = model(full_view, conf=DETECT_CONF, classes=[PERSON_CLASS_ID], iou=0.4,
                         imgsz=640, device=DEVICE, half=HALF_PRECISION, verbose=False)
        detections = sv.Detections(
            xyxy=results[0].boxes.xyxy.cpu().numpy(),
            confidence=results[0].boxes.conf.cpu().numpy(),
            class_id=results[0].boxes.cls.cpu().numpy().astype(int),
        )
        kpts = results[0].keypoints
        if kpts is not None and kpts.conf is not None and len(detections.xyxy) > 0:
            kp_xy = kpts.xy.cpu().numpy()
            kp_cf = kpts.conf.cpu().numpy()
        else:
            kp_xy = np.zeros((len(detections.xyxy), 17, 2))
            kp_cf = np.zeros((len(detections.xyxy), 17))

        n_det = len(detections.xyxy)
        if n_det > 0:
            dets = np.hstack([detections.xyxy, detections.confidence[:, None],
                               detections.class_id[:, None].astype(float)])
        else:
            dets = np.empty((0, 6))
        tracks = self.botsort.update(dets, full_view)
        tracker_ids = [None] * n_det
        if tracks is not None and len(tracks) > 0:
            for tr in tracks:
                j = int(tr[7])
                if 0 <= j < n_det:
                    tracker_ids[j] = int(tr[4])

        dx, dy = self.delta
        kept = []
        for i in range(n_det):
            bbox_full = detections.xyxy[i]
            tid = tracker_ids[i]
            core_bbox = bbox_full + np.array([dx, dy, dx, dy])
            ccx, ccy = (core_bbox[0] + core_bbox[2]) / 2, (core_bbox[1] + core_bbox[3]) / 2
            in_core = (0 <= ccx < self.w) and (0 <= ccy < self.h)
            global_center = ((bbox_full[0] + bbox_full[2]) / 2 + self.fox,
                              (bbox_full[1] + bbox_full[3]) / 2 + self.foy)

            if self.mirror_polys and _feet_in_polys(core_bbox[np.newaxis, :], self.mirror_polys)[0]:
                continue

            is_beige = beige_at_keypoints(full_view, kp_xy[i], kp_cf[i])
            is_white = white_at_keypoints(full_view, kp_xy[i], kp_cf[i]) if is_beige else False
            role = "SELLER" if (is_beige and is_white) else "CLIENT"
            if seller_zone_frac(core_bbox, self.seller_polys) >= SELLER_ZONE_FRAC and \
                    white_at_keypoints(full_view, kp_xy[i], kp_cf[i]):
                role = "SELLER"

            if tid is not None and tid in self.entry_by_tid and role == "SELLER":
                self.entry_seller_secs[tid] = self.entry_seller_secs.get(tid, 0.0) + SKIP / fps
                if self.entry_seller_secs[tid] >= REVERT_MIN_SELLER_SECS and self.entry_count_local > 0:
                    self.entry_count_local -= 1
                    del self.entry_by_tid[tid]
                    self.entry_seller_secs.pop(tid, None)

            if role == "CLIENT" and in_core and box_overlap_frac(core_bbox, self.ignore_polys) > IGNORE_ZONE_FRAC:
                continue

            state_kp_idx = LOWER_KP_IDX + UPPER_KP_IDX
            confident = float(np.mean(kp_cf[i][state_kp_idx])) >= STATE_CONF_MIN
            kept.append(dict(bbox_full=bbox_full, core_bbox=core_bbox, tid=tid, role=role,
                              kp_xy=kp_xy[i], kp_cf=kp_cf[i], confident=confident,
                              global_center=global_center, in_core=in_core))

        embeddings = [None] * len(kept)
        if kept:
            boxes_arr = np.array([k['bbox_full'] for k in kept], dtype=float)
            feats = extract_embeddings(self.botsort, boxes_arr, full_view)
            if feats is not None and feats.ndim == 2 and feats.shape[0] == len(kept) and feats.shape[1] > 1:
                feats = feats / (np.linalg.norm(feats, axis=1, keepdims=True) + 1e-9)
                for i, k in enumerate(kept):
                    if k['confident']:
                        embeddings[i] = feats[i]

        persons_core: List[Person] = []
        for k, emb in zip(kept, embeddings):
            gid = motion_mgr.resolve(self.camera_id, k['tid'], k['global_center'], emb, timestamp)
            if k['in_core']:
                persons_core.append(Person(bbox=k['core_bbox'], role=k['role'], tid=k['tid'],
                                            global_id=gid, kp_xy=k['kp_xy'], kp_cf=k['kp_cf']))

        # --- entry: id-free ordered two-zone pass (CORE-local), deduped globally ---
        for p in persons_core:
            if p.role != "CLIENT":
                continue
            f1 = box_overlap_frac(p.bbox, self.enter1)
            f2 = box_overlap_frac(p.bbox, self.enter2)
            cz = 0
            if f1 >= ENTER_ZONE_FRAC or f2 >= ENTER_ZONE_FRAC:
                cz = 1 if f1 >= f2 else 2
            cx, cy = (p.bbox[0] + p.bbox[2]) / 2.0, (p.bbox[1] + p.bbox[3]) / 2.0
            pf = processed_frame_idx
            best, best_dist = None, ZONE_MATCH_DIST
            for o in self.zone_objs:
                if pf - o[4] <= ZONE_TTL:
                    dd = np.hypot(cx - o[0], cy - o[1])
                    if dd < best_dist:
                        best_dist, best = dd, o
            if best is None:
                if cz != 0:
                    self.zone_objs.append([cx, cy, cz, False, pf])
            else:
                prev_zone = best[2]
                if cz != 0 and cz != prev_zone:
                    if prev_zone == 1 and cz == 2 and not best[3]:
                        self.entry_count_local += 1
                        best[3] = True
                        if p.tid is not None:
                            self.entry_by_tid[p.tid] = timestamp
                        if p.global_id not in global_entered_ids:
                            global_entered_ids.add(p.global_id)
                    best[2] = cz
                best[0], best[1], best[4] = cx, cy, pf
        self.zone_objs = [o for o in self.zone_objs if processed_frame_idx - o[4] <= ZONE_TTL]
        self.entry_by_tid = {t: ts for t, ts in self.entry_by_tid.items() if timestamp - ts <= ENTRY_REVERT_SECS}
        self.entry_seller_secs = {t: s for t, s in self.entry_seller_secs.items() if t in self.entry_by_tid}

        # --- interaction / PEC (CORE-local proximity), deduped globally ---
        grace_frames = max(1, round(INTERACTION_GRACE_SECS * fps / SKIP))
        seller_centers = [((b.bbox[0] + b.bbox[2]) / 2, (b.bbox[1] + b.bbox[3]) / 2)
                           for b in persons_core if b.role == "SELLER"]
        pf = processed_frame_idx
        for p in persons_core:
            if p.role != "CLIENT":
                continue
            cc = ((p.bbox[0] + p.bbox[2]) / 2, (p.bbox[1] + p.bbox[3]) / 2)
            close = any(circle_overlap_frac(cc, sc, INTERACTION_RADIUS) >= INTERACTION_OVERLAP_MIN
                        for sc in seller_centers)
            if close:
                last = self.contact_last_frame.get(p.global_id)
                if last is not None and pf - last <= grace_frames:
                    self.contact_secs[p.global_id] = self.contact_secs.get(p.global_id, 0.0) + SKIP / fps
                else:
                    self.contact_secs[p.global_id] = SKIP / fps
                self.contact_last_frame[p.global_id] = pf
                if self.contact_secs[p.global_id] >= INTERACTION_MIN_SECS:
                    for s in persons_core:
                        if s.role == "SELLER":
                            pair = (s.global_id, p.global_id)
                            if pair not in global_pec_pairs:
                                global_pec_pairs.add(pair)
                                served_gids.add(p.global_id)
                    self.contact_secs.pop(p.global_id, None)
                    self.contact_last_frame.pop(p.global_id, None)
        for gid in list(self.contact_last_frame):
            if pf - self.contact_last_frame[gid] > grace_frames:
                self.contact_secs.pop(gid, None)
                self.contact_last_frame.pop(gid, None)

        draw_frame(core_view, persons_core, polys, served_gids)
        self.last_core_persons = persons_core


# =====================================================================
# Main
# =====================================================================
def main():
    if torch.cuda.is_available():
        print(f"[GPU] {torch.cuda.get_device_name(0)} | CUDA {torch.version.cuda}")
    else:
        print("[GPU] no CUDA — running on CPU (slower, still works).")

    model = YOLO(MODEL_NAME)
    model.to(DEVICE)
    model.fuse()

    video_info = sv.VideoInfo.from_video_path(SOURCE_VIDEO_PATH)
    fps = video_info.fps
    core_regions = split_regions(video_info.width, video_info.height, GRID_ROWS, GRID_COLS)
    adjacency = build_adjacency(GRID_ROWS, GRID_COLS)
    full_regions = build_full_regions(core_regions, adjacency, OVERLAP_PX, video_info.width, video_info.height)

    cameras = [CameraWorker(i, c_o, c_s, f_o, f_s) for i, ((c_o, c_s), (f_o, f_s))
               in enumerate(zip(core_regions, full_regions))]

    motion_mgr = MotionIDManager(adjacency, POSITION_TOLERANCE_PX, HANDOFF_WINDOW_SEC, VEL_EMA_ALPHA,
                                  EMBEDDING_TIEBREAK_MIN, FALLBACK_SIM_THRESHOLD,
                                  GLOBAL_GALLERY_TTL_SEC, GLOBAL_EMA_ALPHA)
    global_entered_ids: set = set()
    global_pec_pairs: set = set()
    served_gids: set = set()

    generator = sv.get_video_frames_generator(SOURCE_VIDEO_PATH)
    processed_frame_idx = -1

    with sv.VideoSink(TARGET_VIDEO_PATH, video_info) as sink:
        for frame_idx, frame in enumerate(generator):
            canvas = frame
            is_skip = (frame_idx % SKIP != 0)
            if not is_skip:
                processed_frame_idx += 1
                if MAX_FRAMES and processed_frame_idx >= MAX_FRAMES:
                    break
            timestamp = frame_idx / fps

            for cam in cameras:
                cam.process(model, canvas, processed_frame_idx, timestamp, fps,
                            motion_mgr, global_entered_ids, global_pec_pairs, served_gids, is_skip)
            if not is_skip:
                motion_mgr.cleanup(timestamp)

            W, H = video_info.width, video_info.height
            cv2.line(canvas, (W // 2, 0), (W // 2, H), (80, 80, 80), 1, cv2.LINE_AA)
            cv2.line(canvas, (0, H // 2), (W, H // 2), (80, 80, 80), 1, cv2.LINE_AA)
            cv2.putText(canvas, f"Entries: {len(global_entered_ids)}  Interactions: {len(global_pec_pairs)}",
                        (30, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

            sink.write_frame(canvas)

            if not is_skip and processed_frame_idx % 100 == 0:
                print(f"[progress] frame {processed_frame_idx} (t={timestamp:.1f}s) | "
                      f"entries={len(global_entered_ids)} interactions={len(global_pec_pairs)}")

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    cross_cam = motion_mgr.cross_camera_identities()
    print("\n========== MCMT RESULT (motion-prediction method) ==========")
    print(f"Unique clients entered (global, deduped) : {len(global_entered_ids)}")
    print(f"PEC / interactions (global, deduped)      : {len(global_pec_pairs)}")
    print(f"Identities confirmed across >1 camera     : {len(cross_cam)} -> {cross_cam[:20]}")
    for cam in cameras:
        print(f"  camera {cam.camera_id} raw local entry crossings: {cam.entry_count_local}")
    print(f"Annotated output video: {TARGET_VIDEO_PATH}")
    print("==============================================================")


if __name__ == "__main__":
    # export_quadrant_reference_frames(frame_number=0)   # run once to calibrate zones, then comment out
    main()

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
[DEVICE] running on cuda:0
[GPU] Tesla T4 | CUDA 12.8
YOLO11m-pose summary (fused): 134 layers, 20,889,004 parameters, 0 gradients, 71.5 GFLOPs


SUCCESS  | BotSort: det_thresh=0.3, max_age=30, max_obs=50, min_hits=3, iou_threshold=0.3, per_class=False, asso_func=iou, track_high_thresh=0.4, track_low_thresh=0.1, new_track_thresh=0.2, track_buffer=180, match_thresh=0.8, proximity_thresh=0.8, appearance_thresh=0.3, cmc_method=sof, frame_rate=30, fuse_first_associate=False, with_reid=True
INFO     | BoxMOT v16.0.11 🚀 Python-3.12.13 torch-2.10.0+cu128
CUDA:0 (Tesla T4, 14912MiB)
INFO     | /usr/local/lib/python3.12/dist-packages/models/osnet_ain_x1_0_msmt17.pt
INFO     | [PID 58] Downloading ReID weights from https://drive.google.com/uc?id=1SigwBE6mPdqiJMqhuIY4aqC7--5CsMal → /usr/local/lib/python3.12/dist-packages/models/osnet_ain_x1_0_msmt17.pt
Downloading...
From: https://drive.google.com/uc?id=1SigwBE6mPdqiJMqhuIY4aqC7--5CsMal
To: /usr/local/lib/python3.12/dist-packages/models/osnet_ain_x1_0_msmt17.pt
100%|██████████| 17.3M/17.3M [00:00<00:00, 75.0MB/s]
SUCCESS  | Loaded pretrained weights from /usr/local/lib/python3.12/dist-pack

WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
[progress] frame 0 (t=0.0s) | entries=0 interactions=0
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and

[hevc @ 0x2a708cc0] Could not find ref with POC 8
[hevc @ 0x2a708cc0] Error constructing the frame RPS.
[hevc @ 0x2ad74500] Could not find ref with POC 9
[hevc @ 0x2ad74500] Error constructing the frame RPS.


WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.


...

WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.


[hevc @ 0x2ad74500] Could not find ref with POC 46
[hevc @ 0x2ad74500] Error constructing the frame RPS.
[hevc @ 0x29f24600] Could not find ref with POC 47
[hevc @ 0x29f24600] Error constructing the frame RPS.


WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.


[hevc @ 0x29f6fbc0] Could not find ref with POC 48
[hevc @ 0x29f6fbc0] Error constructing the frame RPS.
[hevc @ 0x2a708cc0] Could not find ref with POC 5
[hevc @ 0x2a708cc0] Error constructing the frame RPS.
[hevc @ 0x2ad74500] Could not find ref with POC 6
[hevc @ 0x2ad74500] Error constructing the frame RPS.
[hevc @ 0x29f24600] Could not find ref with POC 7
[hevc @ 0x29f24600] Error constructing the frame RPS.
[hevc @ 0x29f6fbc0] Could not find ref with POC 8
[hevc @ 0x29f6fbc0] Error constructing the frame RPS.
[hevc @ 0x2a708cc0] Could not find ref with POC 9
[hevc @ 0x2a708cc0] Error constructing the frame RPS.
[hevc @ 0x2ad74500] Could not find ref with POC 10
[hevc @ 0x2ad74500] Error constructing the frame RPS.
[hevc @ 0x29f24600] Could not find ref with POC 11
[hevc @ 0x29f24600] Error constructing the frame RPS.
[hevc @ 0x29f6fbc0] Could not find ref with POC 12
[hevc @ 0x29f6fbc0] Error constructing the frame RPS.
[hevc @ 0x2a708cc0] Could not find ref with POC 13
[hevc @ 0

WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
WARNING ⚠️ 'half' is deprecated and will be removed in the future. Use 'quantize' instead.

In [3]:
!ffmpeg -y -i /kaggle/working/SC_mcmt_motion_output.mp4 -vcodec libx264 -crf 28 -preset fast /kaggle/working/SC_mcmt_output_compressed.mp4
!ls -lh /kaggle/working/SC_mcmt_output_compressed.mp4

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab